In [11]:
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_recall_curve, matthews_corrcoef
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, auc
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from collections import Counter
from Bio import SeqIO
import itertools
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support


In [24]:
# Preprocessing Functions
def filter_sequence(sequence):
    """ Filter out ambiguous characters and retain only 'A', 'C', 'G', and 'T'. """
    return ''.join([nuc for nuc in sequence if nuc in 'ACGT'])

def get_kmer_frequencies(sequence, k=4):
    """ Calculate k-mer frequencies, skipping ambiguous nucleotides. """
    sequence = filter_sequence(sequence)
    kmer_counts = Counter([sequence[i:i+k] for i in range(len(sequence) - k + 1)])
    total_kmers = sum(kmer_counts.values())
    return {kmer: count / total_kmers for kmer, count in kmer_counts.items()}

def get_position_specific_kmer_frequencies(sequence, k=4, num_regions=3):
    """ Calculate position-specific k-mer frequencies, skipping ambiguous nucleotides. """
    sequence = filter_sequence(sequence)
    region_size = len(sequence) // num_regions
    feature_vector = []
    
    for region_idx in range(num_regions):
        start = region_idx * region_size
        end = (region_idx + 1) * region_size if region_idx != num_regions - 1 else len(sequence)
        region_seq = sequence[start:end]
        region_kmer_freq = get_kmer_frequencies(region_seq, k=k)
        
        possible_kmers = [''.join(p) for p in itertools.product('ACGT', repeat=k)]
        region_vector = [region_kmer_freq.get(kmer, 0) for kmer in possible_kmers]
        feature_vector.extend(region_vector)
        
    return feature_vector

def nucleotide_composition(sequence):
    """ Calculate nucleotide composition for valid nucleotides only ('A', 'C', 'G', 'T'). """
    composition = {'A': 0, 'C': 0, 'G': 0, 'T': 0}
    valid_sequence = filter_sequence(sequence)
    for nucleotide in valid_sequence:
        composition[nucleotide] += 1
    total = len(valid_sequence)
    return [composition['A'] / total, composition['C'] / total, composition['G'] / total, composition['T'] / total]

def create_hybrid_feature_matrix(sequences, k=4, num_regions=3, use_pca=False, n_components=50):
    """ Combine k-mer and nucleotide composition features. """
    feature_matrix = []
    for seq in sequences:
        kmer_features = get_position_specific_kmer_frequencies(seq, k=k, num_regions=num_regions)
        composition_features = nucleotide_composition(seq)
        hybrid_features = kmer_features + composition_features
        feature_matrix.append(hybrid_features)
    
    feature_matrix = np.array(feature_matrix)
    
    if use_pca:
        pca = PCA(n_components=n_components)
        feature_matrix = pca.fit_transform(feature_matrix)
    
    return feature_matrix

# Function to read sequences from FASTA files with labels embedded in headers
def load_sequences_with_labels_from_fasta(file):
    sequences, labels = [], []
    for record in SeqIO.parse(file, "fasta"):
        label = record.description  # Directly use the description (e.g., 'COVID', 'DENGUE')
        if ' ' in label:
            label = label.split()[0]  # Take the first part if there are spaces
        sequence = filter_sequence(str(record.seq))    # Filter ambiguous characters
        if len(sequence) > 0:  # Ensure sequence is valid after filtering
            sequences.append(sequence)
            labels.append(label)
    return sequences, labels

# Evaluation and Training
from sklearn.metrics import precision_recall_fscore_support

def evaluate_models(X, y):
    models = {
        'KNN': KNeighborsClassifier(),
        'Random Forest': RandomForestClassifier(),
        'SVM': SVC(probability=True),
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Decision Tree': DecisionTreeClassifier(),
        'MLP': MLPClassifier(max_iter=1000),
        'Gradient Boosting': GradientBoostingClassifier()
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    
    for model_name, model in models.items():
        print(f"Evaluating {model_name}...")
        accuracies, precisions, recalls, f1_scores, rocs, auprcs, mccs = [], [], [], [], [], [], []
        
        for train_index, test_index in kf.split(X):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)

            accuracies.append(accuracy_score(y_test, y_pred))
            precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
            precisions.append(precision)
            recalls.append(recall)
            f1_scores.append(f1)
            mccs.append(matthews_corrcoef(y_test, y_pred))

            if len(set(y_test)) == 2:  # Binary classification
                y_prob_binary = y_prob[:, 1]  # Get probabilities for the positive class
                rocs.append(roc_auc_score(y_test, y_prob_binary))
                precision, recall, _ = precision_recall_curve(y_test, y_prob_binary)
                auprcs.append(auc(recall, precision))
            else:  # Multi-class classification
                # Use one-vs-rest probabilities for AUC calculation
                rocs.append(roc_auc_score(y_test, y_prob, multi_class='ovr'))
                
                # Compute precision and recall for multi-class
                precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
                precisions.append(precision)
                recalls.append(recall)
                
                # No AUC calculation here, because precision-recall curves are for binary
                # You can compute AUPRC using the probabilities for each class if needed.
        
        results[model_name] = {
            'Accuracy': np.mean(accuracies),
            'Precision': np.mean(precisions),
            'Recall': np.mean(recalls),
            'F1-Score': np.mean(f1_scores),
            'MCC': np.mean(mccs),
            'ROC AUC': np.mean(rocs),
            'AUPRC': np.nan  # Placeholder or calculation based on a different method if needed
        }
        
    return results

In [25]:
# Main execution
fasta_file = 'all_data.fasta'  # Replace with the path to your small dataset in the mentioned format
sequences, labels = load_sequences_with_labels_from_fasta(fasta_file)
X = create_hybrid_feature_matrix(sequences, k=4, num_regions=3, use_pca=True, n_components=50)

# Encode labels to numerical format
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

In [26]:
# Evaluate the models
results = evaluate_models(X, y)
print("Evaluation Results:")
for model, metrics in results.items():
    print(f"{model}: {metrics}")

Evaluating KNN...
Evaluating Random Forest...
Evaluating SVM...
Evaluating Logistic Regression...
Evaluating Decision Tree...
Evaluating MLP...
Evaluating Gradient Boosting...
Evaluation Results:
KNN: {'Accuracy': 0.9961957949933902, 'Precision': 0.9962516463587653, 'Recall': 0.9961995835014135, 'F1-Score': 0.996201471322156, 'MCC': 0.9952460712117311, 'ROC AUC': 0.9992743071502325, 'AUPRC': nan}
Random Forest: {'Accuracy': 0.9933925909877814, 'Precision': 0.9935088187823145, 'Recall': 0.9933679137507252, 'F1-Score': 0.9933980698704328, 'MCC': 0.9917500646529678, 'ROC AUC': 0.9999431522177125, 'AUPRC': nan}
SVM: {'Accuracy': 0.9965961953937906, 'Precision': 0.9966746353798897, 'Recall': 0.9965811662661332, 'F1-Score': 0.9965985780457534, 'MCC': 0.9957457709135811, 'ROC AUC': 0.9995170959658433, 'AUPRC': nan}
Logistic Regression: {'Accuracy': 0.9741685573348902, 'Precision': 0.9755569681239893, 'Recall': 0.9742163561793209, 'F1-Score': 0.9742881681737959, 'MCC': 0.9679703329097705, 'ROC